# Performance- und Machbarkeitstest: Dijkstra-Router auf großem Netzwerk

Dieses Notebook evaluiert die Leistung und Lösbarkeit des `DijkstraRouter` auf dem großen Netzwerk (`large_network.json`) mit einem Planungshorizont von 30 Tagen und 50.000 zufällig generierten Sendungen.

## Zielsetzung
- Messung der Berechnungsdauer (Gesamtzeit und Zeit pro Sendung).
- Ermittlung der Anzahl und des Prozentanteils unlösbarer Sendungen.
- Auswertung der Konsolidierungsrate (Bündelung von Sendungen auf gemeinsamen Transportwegen).
- Einbindung des Routers direkt aus `heuristics/dijkstra_router.py` gemäß Vorgabe.

## 1. Setup und Imports

In [6]:
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader
from freight_routing.data_models import Shipment, ObjectiveWeights, ArcType
from heuristics.dijkstra_router import DijkstraRouter

## 2. Laden des Netzwerks

In [7]:
network_path = PROJECT_ROOT / "dataset/large_network.json"
print(f"Lade Netzwerk aus {network_path}...")
t0 = time.time()
network_data = NetworkDataLoader.from_json(network_path)
print(f"Netzwerk in {time.time() - t0:.2f} s geladen.")
network_data.summary()

Lade Netzwerk aus /home/phil/studium/4Semester/or/Sustainable_Freight_Mode_Choice/dataset/large_network.json...
Netzwerk in 0.29 s geladen.
Summary NetworkData:
hubs=870
arcs=36272
modes=4


## 3. Generierung von 50.000 zufälligen Sendungen

In [8]:
# Seed für Reproduzierbarkeit
random.seed(42)
N_SHIPMENTS = 50000
PLANNING_DAYS = 30
DEADLINE = PLANNING_DAYS * 24 * 60  # 30 Tage in Minuten (43.200 min)

hubs_list = list(network_data.hubs.keys())
shipments = []
for i in range(N_SHIPMENTS):
    start = random.choice(hubs_list)
    dest = random.choice(hubs_list)
    while dest == start:
        dest = random.choice(hubs_list)

    shipments.append(
        Shipment(
            id=f"shipment_{i}",
            start_hub=start,
            end_hub=dest,
            start_time=0,
            deadline=DEADLINE,
            max_price=1000000.0,
            max_emissions=None,
            weight=float(random.randint(1, 10)),
        )
    )
print(f"{len(shipments)} zufällige Sendungen generiert.")

50000 zufällige Sendungen generiert.


## 4. Durchführung des Performance-Tests

Wir initialisieren den `DijkstraRouter` und führen die sequentielle Routenplanung mittels `solve_multiple` aus.

**WICHTIG ZUR RUNTIME:** Da `solve_multiple` intern einen linearen Scan über alle Knoten im zeitexpandierten Graphen durchführt, benötigt die Berechnung pro Sendung ca. 1.4 Sekunden. Das vollständige Routing aller 50.000 Sendungen würde daher ca. 20 Stunden dauern.

Für einen schnellen Vorabtest ist `LIMIT_RUN = 100` voreingestellt (Rechenzeit: ca. 2.5 Minuten). Sie können diesen Parameter auf `None` setzen oder erhöhen, um größere Mengen zu testen.

In [ ]:
# Ändern Sie dies auf None oder 50000 für den vollständigen Test.
LIMIT_RUN = 30000

if LIMIT_RUN is not None:
    test_shipments = shipments[:LIMIT_RUN]
    print(f"Führe Test für {LIMIT_RUN} Sendungen aus")
else:
    test_shipments = shipments
    print(f"Führe VOLLSTÄNDIGEN Test für alle {N_SHIPMENTS} Sendungen aus (geschätzte Zeit: ~20 Stunden)... ")

weights = ObjectiveWeights(cost=0.4, emissions=0.4, time=0.2)
router = DijkstraRouter(network_data, objective_weights=weights)

t_routing_start = time.time()
routing_result = router.solve_multiple(test_shipments, planning_days=PLANNING_DAYS)
t_routing_duration = time.time() - t_routing_start

print(f"\nTest abgeschlossen!")
print(f"Gesamtdauer für das Routing: {t_routing_duration:.2f} s")

Führe Test für 30000 Sendungen aus


## 5. Auswertung und Analyse

In [16]:
total_tested = len(test_shipments)
solved_count = len(routing_result.shipment_routes)
unsolvable_count = total_tested - solved_count
pct_unsolvable = (unsolvable_count / total_tested) * 100
avg_time_per_shipment = t_routing_duration / total_tested

# Analyse der Konsolidierung
transport_arc_shipments = defaultdict(list)
for s_id, route in routing_result.shipment_routes.items():
    for arc in route:
        if arc.arc_type == ArcType.TRANSPORT:
            transport_arc_shipments[arc].append(s_id)

consolidated_shipments = set()
for arc, s_ids in transport_arc_shipments.items():
    if len(s_ids) >= 2:
        for s_id in s_ids:
            consolidated_shipments.add(s_id)

num_consolidated = len(consolidated_shipments)
pct_consolidated = (num_consolidated / solved_count) * 100 if solved_count > 0 else 0.0

print("============================================================")
print(f"                       Testergebnisse                       ")
print("============================================================")
print(f"Gesamtzahl getesteter Sendungen : {total_tested:,}")
print(f"Berechnungsdauer insgesamt      : {t_routing_duration:.2f} s ({t_routing_duration/60:.2f} min)")
print(f"Durchschnittliche Zeit / Sendung : {avg_time_per_shipment:.6f} s")
print(f"Erfolgreich gelöste Sendungen   : {solved_count:,}")
print(f"Anzahl unlösbarer Sendungen     : {unsolvable_count:,}")
print(f"Prozent unlösbarer Sendungen    : {pct_unsolvable:.2f} %")
print(f"Konsolidierte Sendungen         : {num_consolidated:,} von {solved_count:,}")
print(f"Konsolidierungsrate (gelöst)    : {pct_consolidated:.2f} %")
print(f"Konsolidiert (is_consolidated)  : {routing_result.is_consolidated}")
print("============================================================")

if unsolvable_count > 0:
    print("\nBeispiele für unlösbare Sendungen (Diagnostics):")
    for diag in list(routing_result.diagnostics)[:10]:
        print(f" - {diag}")

                       Testergebnisse                       
Gesamtzahl getesteter Sendungen : 20
Berechnungsdauer insgesamt      : 46.70 s (0.78 min)
Durchschnittliche Zeit / Sendung : 2.335135 s
Erfolgreich gelöste Sendungen   : 19
Anzahl unlösbarer Sendungen     : 1
Prozent unlösbarer Sendungen    : 5.00 %
Konsolidierte Sendungen         : 9 von 19
Konsolidierungsrate (gelöst)    : 47.37 %
Konsolidiert (is_consolidated)  : True

Beispiele für unlösbare Sendungen (Diagnostics):
 - Shipment shipment_2: No feasible path found.
